## Dados da Secretária do Tesouro Nacional - STN e Siconfi
  
Descrição:  

Este script permite baixar a base de dados dos estados brasileiros entre 2015 e 2023 (ou períodos posteriores). O volume de dados é elevado e gera um grande número de observações para cada ente federativo. Importante: os valores são NOMINAIS e exigem deflacionamento e tratamento prévio. Atenção especial à identificação e ao significado técnico de cada variável fiscal é fundamental.  
  
Nota:  
O processo de download e a criação da planilha de dados levam aproximadamente 25 minutos para conclusão.

In [ ]:
# pacotes 

!pip install requests pandas tqdm openpyxl

In [ ]:
import requests
import pandas as pd
from tqdm import tqdm

# Caminho do arquivo e URL da API
excel_path = "{LOCAL_DO_ARQUIVO}/UF_IBGE.xlsx"
url_template = "https://apidatalake.tesouro.gov.br/ords/siconfi/tt/dca?an_exercicio={}&id_ente={}"


# Importante!!!! setar o caminho correto do arquivo 'UF_IBGE.xlsx'

# Carregar o arquivo Excel
df_ibge = pd.read_excel(excel_path)

# Verificar se a coluna 'COD_IBGE' está presente
if 'COD_IBGE' not in df_ibge.columns:
    raise ValueError("Coluna 'COD_IBGE' não encontrada no arquivo UF_IBGE.xlsx")

# Lista de exercícios (anos)
exercicios = range(1997, 2024)

# Lista para armazenar os resultados de todas as requisições
resultados = []

# Loop para percorrer cada ID do IBGE e cada ano, fazendo a requisição na API
for cod_ibge in tqdm(df_ibge['COD_IBGE'], desc="Requisitando dados da API"):
    for ano in exercicios:
        url = url_template.format(ano, cod_ibge)
        try:
            response = requests.get(url)
            if response.status_code == 200:
                data = response.json().get('items', [])
                if data:
                    resultados.extend(data)
            else:
                print(f"Erro na requisição: {response.status_code} para ID_IBGE {cod_ibge} no ano {ano}")
        except Exception as e:
            print(f"Erro ao acessar a API: {e}")

# Converter os resultados para um DataFrame
df_resultados = pd.DataFrame(resultados)


Requisitando dados da API: 100%|██████████| 27/27 [13:30<00:00, 30.01s/it]


In [ ]:
# Salvar o resultado em CSV com separador ';' e codificação UTF-8

output_csv = r'{LOCAL_DO_ARQUIVO}\UF_IBGE.xlsx'
df_resultados.to_csv(output_csv, sep=';', encoding='utf-8', index=False)

print(f"Arquivo CSV salvo em {output_csv}")

# Importante!!!! setar o caminho correto do arquivo 'UF_IBGE.xlsx'

Arquivo CSV salvo em /home/erodrigo/Área de trabalho/Academico/3.4 Working papers_publica/01_paper_2021_01 crise fiscal, receita e despesa_submeter nova-economia-UFMG_mar_2026/data_codigos/01_trata_data\UF_IBGE.xlsx


In [ ]:
## FIM